# Day 3 — Word Embeddings
## 30 Days of AI: From NLP to LLMs

---

Yesterday we represented text as word counts and TF-IDF scores. Those methods work, but they have a fundamental problem: they treat every word as an isolated symbol. The words "car" and "automobile" look completely unrelated to a TF-IDF model even though any human knows they mean the same thing.

Word embeddings solve this. By the end of today you will understand how machines learn meaning from raw text and build a working semantic similarity engine.

---

### What You Will Learn Today

- Why word embeddings exist and the intuition behind them
- The Distributional Hypothesis — the theoretical foundation
- Word2Vec: CBOW and Skip-gram architectures
- GloVe: global co-occurrence statistics
- FastText: character n-grams and OOV handling
- The major limitation of classical embeddings
- Embedding similarity measures

### Goal by End of Day

Build a semantic similarity engine that finds similar words, computes sentence similarity, and compares Word2Vec vs FastText behavior.

In [5]:
# Install everything needed for today (run once)
#!pip install gensim numpy scikit-learn matplotlib -q

---

## Part 1 — Why Word Embeddings Exist

### The Problem with BoW and TF-IDF

Bag of Words and TF-IDF represent text as vectors of word counts or weighted frequencies. They work reasonably well for classification, but they have one critical flaw: **they treat every word as an independent symbol with no relationship to any other word**.

Consider this:

```
Sentence A:  "I need to book a flight"
Sentence B:  "I need to reserve a plane ticket"
```

These two sentences mean almost exactly the same thing. But in a TF-IDF model they share almost no vocabulary — "book" vs "reserve", "flight" vs "plane ticket" — so their vectors would look very different.

Word embeddings solve this by **mapping words into a continuous dense vector space where meaning is encoded as position**. Words with similar meanings end up close together in this space.

What embeddings allow models to capture:
- **Similarity** — "dog" and "cat" are close; "dog" and "justice" are far
- **Analogy** — relationships like gender, tense, and geography are encoded as directions
- **Context relationships** — words that appear in similar situations get similar vectors

In [3]:
# Demonstrating the BoW limitation before embeddings solve it
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "I need to book a flight",
    "I need to reserve a plane ticket",   # same meaning, different words
    "The weather is sunny today"           # completely different meaning
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)
similarity = cosine_similarity(tfidf_matrix)

print("TF-IDF Cosine Similarity Matrix")
print("=" * 45)
print(f"{'':30} Sent-1   Sent-2   Sent-3")
labels = ["Book a flight", "Reserve a plane ticket", "Sunny weather"]
for i, row in enumerate(similarity):
    scores = "   ".join([f"{v:.3f}" for v in row])
    print(f"{labels[i]:<30} {scores}")

print()
print("Observation:")
print("  Sentence 1 vs Sentence 2 similarity:", round(similarity[0][1], 3))
print("  These mean the same thing but TF-IDF sees almost no overlap.")
print("  Word embeddings fix this problem.")

TF-IDF Cosine Similarity Matrix
                               Sent-1   Sent-2   Sent-3
Book a flight                  1.000   0.319   0.000
Reserve a plane ticket         0.319   1.000   0.000
Sunny weather                  0.000   0.000   1.000

Observation:
  Sentence 1 vs Sentence 2 similarity: 0.319
  These mean the same thing but TF-IDF sees almost no overlap.
  Word embeddings fix this problem.


---

## Part 2 — The Distributional Hypothesis

The entire field of word embeddings rests on one foundational idea, first stated by linguist J.R. Firth in 1957:

> **"You shall know a word by the company it keeps."**

In modern terms: **words that appear in similar contexts have similar meanings**.

Think about how "dog" and "cat" appear in text:

```
"The dog ran across the yard"
"The cat ran across the yard"

"She fed her dog every morning"
"She fed her cat every morning"

"He took the dog to the vet"
"He took the cat to the vet"
```

Both words appear in nearly identical contexts — yards, feeding, vets, owners. A model that learns from co-occurrence will naturally give "dog" and "cat" similar vector representations, without ever being told they are both animals.

This is the core insight that makes word embeddings work. **Meaning is not defined — it is learned from patterns of use.**

This is also why embeddings reflect human biases present in training text. If the corpus says "nurse" and "she" co-occur frequently, the model learns that association. Embeddings do not know what is true — only what is statistically common.

---

## Part 3 — Word2Vec

Word2Vec (Mikolov et al., 2013) was the breakthrough paper that made word embeddings practical. It trains a shallow neural network to predict words from context, and the learned weights become the embeddings.

### CBOW — Continuous Bag of Words

CBOW predicts the **target word** from its surrounding **context words**.

```
Input context:  ["I", "love", "___", "and", "it"]
Predict:         "NLP"
```

The model averages the context word vectors and tries to predict the center word. Because it averages, it smooths over individual word signals.

- Faster to train
- Works well when you have large amounts of data
- Less effective for rare words

### Skip-gram

Skip-gram does the opposite — it predicts **context words** from a single **target word**.

```
Input target:   "NLP"
Predict:        ["I", "love", "and", "it"]
```

For each target word the model generates multiple training pairs, one per context word. This means rare words get many learning opportunities.

- Slower to train
- Captures semantics more strongly
- Performs significantly better for rare words and small datasets

> **Interview insight:** For semantic similarity tasks, prefer Skip-gram. For fast training on large clean corpora, CBOW is sufficient.

### The Famous Analogy Property

The most striking thing Word2Vec demonstrated was that vector arithmetic can encode semantic relationships:

```
king - man + woman  ≈  queen
paris - france + italy  ≈  rome
walked - walk + swim  ≈  swam
```

This works because the embedding space learns semantic directions. The direction from "man" to "woman" is similar to the direction from "king" to "queen". Gender, tense, geography, and many other relationships become spatial directions in the vector space.

In [6]:
# Train a Word2Vec model from scratch on a small corpus
from gensim.models import Word2Vec
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

# Small corpus — in production you would use millions of sentences
corpus = [
    "the cat sat on the mat",
    "the dog sat on the floor",
    "cats and dogs are common pets",
    "the king ruled the kingdom with wisdom",
    "the queen governed the land wisely",
    "man and woman are both human beings",
    "deep learning is a subset of machine learning",
    "machine learning is a subset of artificial intelligence",
    "neural networks power deep learning models",
    "NLP is the study of natural language processing",
    "natural language processing enables machines to understand text",
    "word embeddings represent words as dense vectors",
    "dense vectors capture semantic meaning of words",
    "python is a popular programming language for data science",
    "data science uses statistics and machine learning",
    "the car drove down the road quickly",
    "the automobile moved along the highway",
    "the dog ran across the yard happily",
    "the cat ran across the yard quickly",
    "she fed her dog every morning before work",
    "she fed her cat every morning at home"
]

# Tokenize corpus
tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]

# Train Word2Vec — Skip-gram (sg=1), CBOW (sg=0)
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,    # embedding dimensions
    window=3,          # context window size
    min_count=1,       # include all words
    sg=1,              # 1 = Skip-gram, 0 = CBOW
    epochs=200,
    seed=42
)

w2v_cbow = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,              # CBOW
    epochs=200,
    seed=42
)

print("Both Word2Vec models trained successfully.")
print(f"Vocabulary size: {len(w2v_skipgram.wv)} words")
print(f"Embedding dimensions: {w2v_skipgram.vector_size}")

Both Word2Vec models trained successfully.
Vocabulary size: 90 words
Embedding dimensions: 50


In [7]:
# Explore what Word2Vec learned — similarity
print("Word Similarity — Skip-gram Model")
print("=" * 45)

pairs = [
    ("cat", "dog"),
    ("cat", "mat"),
    ("machine", "learning"),
    ("king", "queen"),
    ("car", "automobile"),
    ("nlp", "language")
]

for word1, word2 in pairs:
    try:
        sim = w2v_skipgram.wv.similarity(word1, word2)
        print(f"  {word1:<15} vs  {word2:<15}  similarity: {sim:.4f}")
    except KeyError as e:
        print(f"  Word not in vocabulary: {e}")

Word Similarity — Skip-gram Model
  cat             vs  dog              similarity: 0.9936
  cat             vs  mat              similarity: 0.9777
  machine         vs  learning         similarity: 0.9936
  king            vs  queen            similarity: 0.9849
  car             vs  automobile       similarity: 0.9882
  nlp             vs  language         similarity: 0.9914


In [8]:
# Most similar words
print("Most Similar Words — Skip-gram Model")
print("=" * 45)

query_words = ["cat", "learning", "king"]

for word in query_words:
    try:
        similar = w2v_skipgram.wv.most_similar(word, topn=4)
        print(f"\nWords most similar to '{word}':")
        for similar_word, score in similar:
            print(f"  {similar_word:<20}  score: {score:.4f}")
    except KeyError as e:
        print(f"  Word not in vocabulary: {e}")

Most Similar Words — Skip-gram Model

Words most similar to 'cat':
  her                   score: 0.9960
  she                   score: 0.9947
  every                 score: 0.9944
  as                    score: 0.9943

Words most similar to 'learning':
  is                    score: 0.9953
  of                    score: 0.9951
  a                     score: 0.9942
  deep                  score: 0.9941

Words most similar to 'king':
  kingdom               score: 0.9948
  science               score: 0.9946
  language              score: 0.9942
  with                  score: 0.9942


In [9]:
# Analogy arithmetic — king - man + woman = ?
print("Analogy Reasoning — Vector Arithmetic")
print("=" * 45)
print("Formula: king - man + woman = ?")
print()

try:
    result = w2v_skipgram.wv.most_similar(
        positive=["king", "woman"],
        negative=["man"],
        topn=5
    )
    print("Top results for: king - man + woman")
    for word, score in result:
        print(f"  {word:<20}  score: {score:.4f}")

    print()
    print("Note: With a small training corpus, results are approximate.")
    print("Pre-trained models on billions of words produce 'queen' reliably.")
except KeyError as e:
    print(f"Word not found: {e}")

Analogy Reasoning — Vector Arithmetic
Formula: king - man + woman = ?

Top results for: king - man + woman
  with                  score: 0.9907
  kingdom               score: 0.9907
  wisdom                score: 0.9902
  wisely                score: 0.9899
  morning               score: 0.9891

Note: With a small training corpus, results are approximate.
Pre-trained models on billions of words produce 'queen' reliably.


---

## Part 4 — GloVe (Global Vectors)

Word2Vec learns embeddings by sliding a window across the text and making local predictions. It never looks at the entire corpus at once — it only sees a few words at a time.

GloVe (Pennington et al., 2014) takes a different approach. It first builds a **global co-occurrence matrix** — a table counting how often every word pair appears together across the entire corpus — then factorizes this matrix to produce embeddings.

```
Word2Vec  =  predictive model (learns from local context windows)
GloVe     =  count-based model (learns from global co-occurrence statistics)
```

### Why This Matters

GloVe captures both local context (nearby word relationships) and global structure (how words relate across the entire corpus). This often makes GloVe embeddings slightly better for downstream NLP tasks, though the difference is task-dependent.

In practice both Word2Vec and GloVe produce high-quality embeddings. Pre-trained GloVe vectors (trained on 840 billion tokens from Common Crawl) are widely used as a starting point for NLP models.

> **Interview insight:** Word2Vec is a predictive model — it learns by predicting words. GloVe is a count-based model — it learns by factorizing co-occurrence counts. Both learn similar representations but via different objectives.

In [10]:
# Demonstrate the GloVe idea by building a co-occurrence matrix manually
from collections import defaultdict
import numpy as np

sample_sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "floor"],
    ["cats", "and", "dogs", "are", "pets"],
]

window_size = 2
cooccurrence = defaultdict(lambda: defaultdict(int))

for sentence in sample_sentences:
    for i, word in enumerate(sentence):
        start = max(0, i - window_size)
        end   = min(len(sentence), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                cooccurrence[word][sentence[j]] += 1

# Show co-occurrence counts for a few words
focus_words = ["cat", "dog", "sat"]
print("Co-occurrence Counts (window=2)")
print("GloVe builds embeddings by factorizing a matrix like this")
print("=" * 50)
for word in focus_words:
    print(f"\n'{word}' co-occurs with:")
    sorted_cooc = sorted(cooccurrence[word].items(), key=lambda x: -x[1])
    for context_word, count in sorted_cooc:
        print(f"  '{context_word}'  ->  {count} time(s)")

print()
print("Observation: 'cat' and 'dog' share context words 'sat', 'the', 'on'.")
print("GloVe uses this global pattern to make their vectors similar.")

Co-occurrence Counts (window=2)
GloVe builds embeddings by factorizing a matrix like this

'cat' co-occurs with:
  'the'  ->  1 time(s)
  'sat'  ->  1 time(s)
  'on'  ->  1 time(s)

'dog' co-occurs with:
  'the'  ->  1 time(s)
  'sat'  ->  1 time(s)
  'on'  ->  1 time(s)

'sat' co-occurs with:
  'the'  ->  4 time(s)
  'on'  ->  2 time(s)
  'cat'  ->  1 time(s)
  'dog'  ->  1 time(s)

Observation: 'cat' and 'dog' share context words 'sat', 'the', 'on'.
GloVe uses this global pattern to make their vectors similar.


---

## Part 5 — FastText and OOV Handling

Both Word2Vec and GloVe have a hard limitation: **they can only represent words seen during training**. If your model encounters a word it has never seen — a typo, a domain-specific term, a new proper noun — it has no vector for it. This is the Out-of-Vocabulary (OOV) problem.

FastText (Bojanowski et al., 2017, Facebook AI) solves this by representing words as **character n-grams** rather than whole tokens.

### How FastText Works

Instead of learning one vector per word, FastText learns vectors for all character substrings of length 3 to 6:

```
Word:      "playing"
N-grams:   "<pl", "pla", "lay", "ayi", "yin", "ing", "ng>"
           (< and > mark word boundaries)

The word vector = sum of all its n-gram vectors
```

### Why This Solves OOV

For a word never seen in training, FastText can still build a reasonable vector by summing the n-gram vectors of its character substrings. Those substrings were likely seen in other words.

Example: If "transformerize" is OOV, FastText uses substrings like "trans", "form", "form", "ize" which appeared in "transform", "formation", "realize" — all known words.

### Additional Advantages

- Captures morphology naturally — "play", "playing", "played", "player" share n-grams and get similar vectors
- Works much better for languages with rich morphology (German, Finnish, Turkish)
- Better representations for rare words even when they are in-vocabulary

> **Key insight:** FastText is a conceptual bridge between classical word embeddings and the subword tokenization used in modern transformers. Both approaches decompose words into meaningful subparts.

In [11]:
from gensim.models import FastText

# Train FastText on the same corpus as Word2Vec
ft_model = FastText(
    sentences=tokenized_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,           # Skip-gram
    min_n=3,        # minimum n-gram length
    max_n=6,        # maximum n-gram length
    epochs=200,
    seed=42
)

print("FastText model trained.")
print(f"Vocabulary size: {len(ft_model.wv)} words")

# Show n-grams for a word
word = "playing"
n = 3
ngrams = [word[i:i+n] for i in range(len(word)-n+1)]
print(f"\nCharacter {n}-grams for '{word}':")
print(" ", ngrams)

FastText model trained.
Vocabulary size: 90 words

Character 3-grams for 'playing':
  ['pla', 'lay', 'ayi', 'yin', 'ing']


In [12]:
import numpy as np

# FastText handles OOV words — Word2Vec cannot
oov_words = ["transformerize", "machinelearning", "NLPmodel", "deeplearner"]

print("OOV Word Handling Comparison")
print("=" * 55)

for word in oov_words:
    word_lower = word.lower()

    # FastText — always produces a vector
    ft_vec  = ft_model.wv[word_lower]
    ft_norm = np.linalg.norm(ft_vec)

    # Word2Vec — KeyError for OOV
    try:
        w2v_vec  = w2v_skipgram.wv[word_lower]
        w2v_result = f"found  (norm={np.linalg.norm(w2v_vec):.3f})"
    except KeyError:
        w2v_result = "KeyError — OOV, no vector"

    print(f"\nWord: '{word}'")
    print(f"  Word2Vec  : {w2v_result}")
    print(f"  FastText  : vector produced  (norm={ft_norm:.3f})")

OOV Word Handling Comparison

Word: 'transformerize'
  Word2Vec  : KeyError — OOV, no vector
  FastText  : vector produced  (norm=0.051)

Word: 'machinelearning'
  Word2Vec  : KeyError — OOV, no vector
  FastText  : vector produced  (norm=1.152)

Word: 'NLPmodel'
  Word2Vec  : KeyError — OOV, no vector
  FastText  : vector produced  (norm=0.277)

Word: 'deeplearner'
  Word2Vec  : KeyError — OOV, no vector
  FastText  : vector produced  (norm=0.348)


---

## Part 6 — The Major Limitation of Classical Embeddings

Word2Vec, GloVe, and FastText all share one fundamental problem: **they assign one fixed vector per word, regardless of context**.

This is called the **polysemy problem**. Many words have multiple meanings depending on context:

```
"I deposited money at the bank"          -- bank = financial institution
"We had a picnic on the river bank"      -- bank = edge of a river
"The pilot had to bank the aircraft"     -- bank = tilt while turning
```

In Word2Vec, "bank" has a single vector — some average of all its meanings across the training corpus. The model cannot distinguish between financial bank and river bank based on context.

### Why This Matters

In tasks like machine translation, question answering, and reading comprehension, the meaning of a word depends entirely on its context. A model with fixed word vectors will make systematic errors on ambiguous words.

### How Transformers Solve This

Transformers like BERT produce **contextual embeddings** — the vector for a word changes depending on the words around it. The same word "bank" gets a different vector in a finance context vs a nature context. This is what makes BERT and GPT dramatically more powerful than classical embedding models for most NLP tasks.

We will cover contextual embeddings in detail when we get to the Transformer architecture.

In [13]:
# Demonstrate the polysemy limitation
# In classical embeddings, 'bank' has ONE vector regardless of context

polysemy_corpus = [
    "i deposited money at the bank yesterday",
    "the bank account has a high interest rate",
    "we sat on the river bank and watched the water",
    "the bank of the river was muddy after rain",
    "the pilot had to bank the plane sharply",
]

polysemy_tokens = [s.split() for s in polysemy_corpus]

model_poly = Word2Vec(
    sentences=polysemy_tokens,
    vector_size=10,
    window=4,
    min_count=1,
    sg=1,
    epochs=300,
    seed=42
)

bank_vector = model_poly.wv["bank"]

print("Polysemy Limitation — Classical Embeddings")
print("=" * 50)
print()
print("Sentences in training data:")
for s in polysemy_corpus:
    print(f"  {s}")

print()
print(f"Word2Vec vector for 'bank' (first 5 dims): {bank_vector[:5].round(4)}")
print()
print("This single vector tries to represent all three meanings of 'bank':")
print("  1. Financial institution")
print("  2. River bank")
print("  3. Aircraft banking")
print()
print("Result: the vector is an average — it fully represents none of them.")
print("Transformers solve this with contextual embeddings. (Coming in Day 7.)")

Polysemy Limitation — Classical Embeddings

Sentences in training data:
  i deposited money at the bank yesterday
  the bank account has a high interest rate
  we sat on the river bank and watched the water
  the bank of the river was muddy after rain
  the pilot had to bank the plane sharply

Word2Vec vector for 'bank' (first 5 dims): [0.075  0.1799 0.0506 0.2097 0.5332]

This single vector tries to represent all three meanings of 'bank':
  1. Financial institution
  2. River bank
  3. Aircraft banking

Result: the vector is an average — it fully represents none of them.
Transformers solve this with contextual embeddings. (Coming in Day 7.)


---

## Part 7 — Embedding Similarity Measures

Once we have word vectors, we need a way to measure how similar two vectors are. There are three common measures:

### Cosine Similarity

Measures the **angle** between two vectors, not their magnitude. Two vectors pointing in the same direction have cosine similarity of 1, perpendicular vectors have 0, and opposite directions give -1.

```
cosine_similarity(A, B) = (A . B) / (||A|| * ||B||)
```

This is the standard measure for word embeddings because it is scale-invariant. A word that appears 100 times gets a larger magnitude vector than one that appears 5 times, but cosine similarity normalizes this away. We care about **direction** (meaning), not **magnitude** (frequency).

### Euclidean Distance

Measures the straight-line distance between two points in vector space. Intuitive but sensitive to vector magnitude. Less preferred for word embeddings for exactly that reason.

### Dot Product

The raw dot product combines both direction and magnitude. Used in attention mechanisms within Transformers where magnitude carries useful signal.

> **Interview insight:** For comparing word or sentence meaning, always use cosine similarity. It measures semantic orientation — how aligned two meanings are — rather than how large the vectors are.

In [14]:
import numpy as np

def cosine_similarity(vec_a, vec_b):
    dot_product = np.dot(vec_a, vec_b)
    norm_a      = np.linalg.norm(vec_a)
    norm_b      = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

def euclidean_distance(vec_a, vec_b):
    return np.linalg.norm(vec_a - vec_b)

def dot_product(vec_a, vec_b):
    return np.dot(vec_a, vec_b)


# Compare all three measures on word pairs
word_pairs = [
    ("cat",     "dog"),
    ("cat",     "learning"),
    ("machine", "learning"),
    ("king",    "queen"),
]

print("Similarity Measures Comparison")
print("=" * 70)
print(f"{'Pair':<30} {'Cosine':>10}  {'Euclidean':>12}  {'Dot Product':>12}")
print("-" * 70)

for w1, w2 in word_pairs:
    try:
        v1  = w2v_skipgram.wv[w1]
        v2  = w2v_skipgram.wv[w2]
        cos = cosine_similarity(v1, v2)
        euc = euclidean_distance(v1, v2)
        dot = dot_product(v1, v2)
        pair_label = f"{w1} vs {w2}"
        print(f"{pair_label:<30} {cos:>10.4f}  {euc:>12.4f}  {dot:>12.4f}")
    except KeyError as e:
        print(f"  Missing word: {e}")

print()
print("Cosine similarity is the most reliable for semantic comparison.")
print("Higher cosine = more similar meaning. Range: -1 to 1.")

Similarity Measures Comparison
Pair                               Cosine     Euclidean   Dot Product
----------------------------------------------------------------------
cat vs dog                         0.9936        0.1423        1.0270
cat vs learning                    0.9907        0.2440        1.1326
machine vs learning                0.9936        0.2508        1.1070
king vs queen                      0.9849        0.1977        0.7434

Cosine similarity is the most reliable for semantic comparison.
Higher cosine = more similar meaning. Range: -1 to 1.


---

## Day 3 Summary

| Concept | Core Idea | Key Limitation |
|---|---|---|
| **Word Embeddings** | Dense vectors that encode meaning | Requires large corpus to learn well |
| **Distributional Hypothesis** | Meaning comes from context patterns | Reflects biases present in training data |
| **CBOW** | Predict target from context | Weaker on rare words |
| **Skip-gram** | Predict context from target | Slower, better for rare words and semantics |
| **GloVe** | Factorize global co-occurrence matrix | Does not handle morphology |
| **FastText** | Character n-grams, handles OOV | Larger model size |
| **Cosine Similarity** | Angle between vectors = semantic alignment | Does not capture magnitude |
| **Polysemy Limitation** | One vector per word regardless of context | Solved by Transformers |

---


*30 Days of AI — Day 3 of 30 | Word Embeddings: Word2Vec, GloVe, FastText*